# Modelado con Scikit-Learn 

Primero se cargaran la totalidad de los datos con PySpark, para luego usar solo la muestra de 1 millón de obervaciones y pasar a pandas el dataset.

In [4]:
from pyspark.sql import SparkSession

# Carga del Dataset 
spark = SparkSession.builder \
    .appName("Avazu_EDA") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.maxResultSize", "0") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()
df_train = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("../data/train.csv")
df_test = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("../data/test.csv")


In [5]:
import pyspark.sql.functions as F
# 1. Convertir 'hour' a String para extracción de caracteres
df_train_1 = df_train.withColumn("hour_str", F.col("hour").cast("string"))

# 2. Extraer día, hora del día y construir la franja horaria directamente
df_transformed = df_train_1 \
    .withColumn("day", F.substring("hour_str", 5, 2).cast("integer")) \
    .withColumn("hour_of_day", F.substring("hour_str", 7, 2).cast("integer")) \
    .withColumn(
        "franja_horaria",
        F.when((F.col("hour_of_day") >= 0) & (F.col("hour_of_day") < 6), "Madrugada")
         .when((F.col("hour_of_day") >= 6) & (F.col("hour_of_day") < 12), "Mañana")
         .when((F.col("hour_of_day") >= 12) & (F.col("hour_of_day") < 18), "Tarde")
         .otherwise("Noche")
    ) \
    .drop("hour_str")

# Mostrar las columnas seleccionadas
df_transformed.select("hour", "day", "hour_of_day", "franja_horaria").show(5)

# transformación para df_test

df_train_2 = df_test.withColumn("hour_str", F.col("hour").cast("string"))
df_transformed_test = df_train_2 \
    .withColumn("day", F.substring("hour_str", 5, 2).cast("integer")) \
    .withColumn("hour_of_day", F.substring("hour_str", 7, 2).cast("integer")) \
    .withColumn(
        "franja_horaria",
        F.when((F.col("hour_of_day") >= 0) & (F.col("hour_of_day") < 6), "Madrugada")
         .when((F.col("hour_of_day") >= 6) & (F.col("hour_of_day") < 12), "Mañana")
         .when((F.col("hour_of_day") >= 12) & (F.col("hour_of_day") < 18), "Tarde")
         .otherwise("Noche")
    ) \
    .drop("hour_str")

+--------+---+-----------+--------------+
|    hour|day|hour_of_day|franja_horaria|
+--------+---+-----------+--------------+
|14102100| 21|          0|     Madrugada|
|14102100| 21|          0|     Madrugada|
|14102100| 21|          0|     Madrugada|
|14102100| 21|          0|     Madrugada|
|14102100| 21|          0|     Madrugada|
+--------+---+-----------+--------------+
only showing top 5 rows



## Muestreo de los datos  

Acá se tomará una muestra estratificada de 1 millón de observaciones del dataset de entrenamiento, con el fin de hacer el modelado con pandas.  

In [6]:
total_rows = df_transformed.count()
fraction = 1_000_000 / total_rows

# Muestreo Estratificado por la variable objetivo 'click' en PySpark
# sampleBy toma un diccionario con las proporciones por cada clase (0 y 1)
df_sample_spark = df_transformed.sampleBy("click", fractions={0: fraction, 1: fraction}, seed=42)

# Convertir la muestra reducida de 1M directamente a un DataFrame de Pandas
datos= df_sample_spark.toPandas()

# Verificación
print(f"Total registros originales: {total_rows:,}")
print(f"Total registros en Pandas: {len(datos):,}")
print("\nDistribución de 'click' en la muestra (Pandas):")
print(datos['click'].value_counts(normalize=True))

Total registros originales: 40,428,967
Total registros en Pandas: 999,336

Distribución de 'click' en la muestra (Pandas):
click
0    0.830516
1    0.169484
Name: proportion, dtype: float64


In [7]:
datos.shape

(999336, 27)

In [8]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999336 entries, 0 to 999335
Data columns (total 27 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                999336 non-null  object
 1   click             999336 non-null  int32 
 2   hour              999336 non-null  int32 
 3   C1                999336 non-null  int32 
 4   banner_pos        999336 non-null  int32 
 5   site_id           999336 non-null  object
 6   site_domain       999336 non-null  object
 7   site_category     999336 non-null  object
 8   app_id            999336 non-null  object
 9   app_domain        999336 non-null  object
 10  app_category      999336 non-null  object
 11  device_id         999336 non-null  object
 12  device_ip         999336 non-null  object
 13  device_model      999336 non-null  object
 14  device_type       999336 non-null  int32 
 15  device_conn_type  999336 non-null  int32 
 16  C14               999336 non-null  int

## Preprocesamiento  

In [9]:
import pandas as pd
import numpy as np
import os
import pickle
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
os.makedirs("../checkpoints", exist_ok=True)

### Selección de variables relevantes  

Se decidió eliminar las variables "id", "hour", "device_ip" y "device_id" por su alta cardinalidad y por ser redundante para el modelo. Además, se definieron las técnicas de preprocesamiento para cada variable: 
- Aplicación de top_k grouping entre 10-50: para las variables "site_id", "site_domain", "app_id", "app_domain", "device_model", "C14"
- Aplicación de top_k grouping entre 10-30: para las variables "C17", "C19", "C20", "C21"
- Aplicación de One_Hot_encoder: para las variables "C1", "banner_pos", "site_category", "app_category","device_type", "device_conn_type", "C15", "C16","C18", "franja_horaria"

In [10]:

cols_drop = ["id", "hour", "device_ip", "device_id"] 

cols_topk_high = ["site_id", "site_domain", "app_id", "app_domain",
                   "device_model", "C14"]        # K entre 10-50

cols_topk_medium = ["C17", "C19", "C20", "C21"]  # K entre 10-30

cols_onehot_direct = ["C1", "banner_pos", "site_category", "app_category",
                       "device_type", "device_conn_type", "C15", "C16",
                       "C18", "franja_horaria"]

cols_numeric = ["day", "hour_of_day"]
target = "click"

df_model = datos.drop(columns=cols_drop).copy()

### Division de train/test estratificada 

In [11]:
X = df_model.drop(columns=[target])
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("\nProporción de clases (train):")
print(y_train.value_counts(normalize=True))
print("\nProporción de clases (test):")
print(y_test.value_counts(normalize=True))

Train: (799468, 22) | Test: (199868, 22)

Proporción de clases (train):
click
0    0.830516
1    0.169484
Name: proportion, dtype: float64

Proporción de clases (test):
click
0    0.830518
1    0.169482
Name: proportion, dtype: float64


In [12]:
X_train.shape

(799468, 22)

### Codificación + escalado

In [13]:
def ajustar_topk_grouping(df_train, columnas, k_dict, categoria_otros="Otros"):
    """Calcula las categorías top-k usando SOLO datos de entrenamiento."""
    top_categorias = {}
    for col in columnas:
        k = k_dict[col]
        # Aseguramos tipo string ANTES de calcular frecuencias
        serie = df_train[col].astype(str)
        top_k = serie.value_counts().nlargest(k).index
        top_categorias[col] = set(top_k)
    return top_categorias

def aplicar_topk_grouping(df, top_categorias, categoria_otros="Otros"):
    """Aplica categorías ya calculadas a cualquier partición (train, test, o test real de Kaggle)."""
    df = df.copy()
    for col, top_k in top_categorias.items():
        df[col] = df[col].astype(str)  # uniformar tipo primero
        df[col] = df[col].where(df[col].isin(top_k), categoria_otros)
    return df

k_dict = {**{c: 30 for c in cols_topk_high}, **{c: 20 for c in cols_topk_medium}}
cols_topk_all = cols_topk_high + cols_topk_medium

# Fit SOLO en train
top_categorias_guardadas = ajustar_topk_grouping(X_train, cols_topk_all, k_dict)

# Transform en train y en test (test usa las categorías aprendidas en train)
X_train = aplicar_topk_grouping(X_train, top_categorias_guardadas)
X_test = aplicar_topk_grouping(X_test, top_categorias_guardadas)



print("\nProporción de clases (train):")
print(y_train.value_counts(normalize=True))
print("\nProporción de clases (test):")
print(y_test.value_counts(normalize=True))



# ============================================================
# Codificación de categóricas + Escalado de numéricas
# ============================================================
cols_categoricas = cols_topk_all + cols_onehot_direct  # topk_high + topk_medium + onehot_direct

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cols_categoricas),
        ("num", StandardScaler(), cols_numeric),
    ],
    remainder="drop"
)

# Fit SOLO en train
X_train_proc = preprocessor.fit_transform(X_train)

# Transform (no fit) en test
X_test_proc = preprocessor.transform(X_test)

print("Dimensión final del vector de entrada:", X_train_proc.shape[1])
print("X_train_proc:", X_train_proc.shape, "| X_test_proc:", X_test_proc.shape)


Proporción de clases (train):
click
0    0.830516
1    0.169484
Name: proportion, dtype: float64

Proporción de clases (test):
click
0    0.830518
1    0.169482
Name: proportion, dtype: float64
Dimensión final del vector de entrada: 370
X_train_proc: (799468, 370) | X_test_proc: (199868, 370)


In [14]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
import time

param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "alpha": [0.0001, 0.001, 0.01],
    "max_iter": [50, 100],
}

checkpoint_dir = "../checkpoints/mlp_gridsearch"
os.makedirs(checkpoint_dir, exist_ok=True)
results_path = os.path.join(checkpoint_dir, "resultados_gridsearch.csv")

# Si ya hay resultados previos, se cargan para no repetir combinaciones
if os.path.exists(results_path):
    resultados_df = pd.read_csv(results_path)
    combinaciones_evaluadas = set(
        zip(resultados_df["hidden_layer_sizes"], resultados_df["alpha"], resultados_df["max_iter"])
    )
else:
    resultados_df = pd.DataFrame(columns=[
        "iteracion", "hidden_layer_sizes", "alpha", "max_iter",
        "accuracy", "precision", "recall", "f1_score", "roc_auc",
        "train_time_seg", "predict_time_seg", "modelo_path"
    ])
    combinaciones_evaluadas = set()

grid = list(ParameterGrid(param_grid))
print(f"Total de combinaciones a evaluar: {len(grid)}")

for i, params in enumerate(grid, start=1):
    combo_key = (str(params["hidden_layer_sizes"]), params["alpha"], params["max_iter"])
    if combo_key in combinaciones_evaluadas:
        print(f"[{i}/{len(grid)}] Ya evaluado, se omite: {params}")
        continue

    print(f"[{i}/{len(grid)}] Entrenando con: {params}")

    modelo = MLPClassifier(
        hidden_layer_sizes=params["hidden_layer_sizes"],
        alpha=params["alpha"],
        max_iter=params["max_iter"],
        random_state=42,
        early_stopping=True,
    )

    t0 = time.time()
    modelo.fit(X_train_proc, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = modelo.predict(X_test_proc)
    y_proba = modelo.predict_proba(X_test_proc)[:, 1]
    predict_time = time.time() - t0

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    modelo_filename = f"mlp_iter{i}.pkl"
    modelo_path = os.path.join(checkpoint_dir, modelo_filename)
    joblib.dump(modelo, modelo_path)
    np.save(os.path.join(checkpoint_dir, f"cm_iter{i}.npy"), cm)

    nueva_fila = {
        "iteracion": i,
        "hidden_layer_sizes": str(params["hidden_layer_sizes"]),
        "alpha": params["alpha"],
        "max_iter": params["max_iter"],
        "accuracy": acc, "precision": prec, "recall": rec,
        "f1_score": f1, "roc_auc": auc,
        "train_time_seg": train_time, "predict_time_seg": predict_time,
        "modelo_path": modelo_path,
    }
    resultados_df = pd.concat([resultados_df, pd.DataFrame([nueva_fila])], ignore_index=True)

    # Checkpoint: se guarda el CSV acumulado tras CADA iteración
    resultados_df.to_csv(results_path, index=False)

    print(f"   -> AUC: {auc:.4f} | F1: {f1:.4f} | Recall: {rec:.4f} | "
          f"Train: {train_time:.1f}s | Predict: {predict_time:.2f}s")

print("\nBúsqueda completada.")

Total de combinaciones a evaluar: 18
[1/18] Ya evaluado, se omite: {'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'max_iter': 50}
[2/18] Entrenando con: {'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'max_iter': 100}
   -> AUC: 0.7331 | F1: 0.1286 | Recall: 0.0725 | Train: 94.8s | Predict: 0.31s
[3/18] Entrenando con: {'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'max_iter': 50}
   -> AUC: 0.7340 | F1: 0.1290 | Recall: 0.0727 | Train: 89.1s | Predict: 0.41s
[4/18] Entrenando con: {'alpha': 0.0001, 'hidden_layer_sizes': (100,), 'max_iter': 100}
   -> AUC: 0.7340 | F1: 0.1290 | Recall: 0.0727 | Train: 86.6s | Predict: 0.39s
[5/18] Entrenando con: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'max_iter': 50}
   -> AUC: 0.7346 | F1: 0.1181 | Recall: 0.0657 | Train: 106.8s | Predict: 0.46s
[6/18] Entrenando con: {'alpha': 0.0001, 'hidden_layer_sizes': (100, 50), 'max_iter': 100}
   -> AUC: 0.7346 | F1: 0.1181 | Recall: 0.0657 | Train: 105.5s | Predict: 0.46s
[7/18] Entrenando con: {'

In [15]:
resultados_df = resultados_df.sort_values("roc_auc", ascending=False)
display(resultados_df.head(10))

mejor_fila = resultados_df.iloc[0]
mejor_modelo = joblib.load(mejor_fila["modelo_path"])

print("Mejor combinación de hiperparámetros:")
print(mejor_fila[["hidden_layer_sizes", "alpha", "max_iter", "roc_auc", "f1_score", "recall"]])

,iteracion,hidden_layer_sizes,alpha,max_iter,accuracy,precision,recall,f1_score,roc_auc,train_time_seg,predict_time_seg,modelo_path
4,5,"(100, 50)",0.0001,50,0.833680,0.582723,0.065714,0.118109,0.734623,106.770765,0.455750,../checkpoints/mlp_gridsearch\mlp_iter5.pkl
5,6,"(100, 50)",0.0001,100,0.833680,0.582723,0.065714,0.118109,0.734623,105.496664,0.459673,../checkpoints/mlp_gridsearch\mlp_iter6.pkl
2,3,"(100,)",0.0001,50,0.833695,0.574061,0.072652,0.128980,0.733972,89.112804,0.411225,../checkpoints/mlp_gridsearch\mlp_iter3.pkl
3,4,"(100,)",0.0001,100,0.833695,0.574061,0.072652,0.128980,0.733972,86.555765,0.390865,../checkpoints/mlp_gridsearch\mlp_iter4.pkl
9,10,"(100,)",0.0010,100,0.833735,0.579599,0.069109,0.123493,0.733957,72.227220,0.387910,../checkpoints/mlp_gridsearch\mlp_iter10.pkl
8,9,"(100,)",0.0010,50,0.833735,0.579599,0.069109,0.123493,0.733957,71.729862,0.406160,../checkpoints/mlp_gridsearch\mlp_iter9.pkl
10,11,"(100, 50)",0.0010,50,0.833820,0.589967,0.063884,0.115284,0.733841,70.638867,0.448594,../checkpoints/mlp_gridsearch\mlp_iter11.pkl
11,12,"(100, 50)",0.0010,100,0.833820,0.589967,0.063884,0.115284,0.733841,72.537413,0.453664,../checkpoints/mlp_gridsearch\mlp_iter12.pkl
6,7,"(50,)",0.0010,50,0.833605,0.578799,0.066895,0.119929,0.733708,104.049732,0.300331,../checkpoints/mlp_gridsearch\mlp_iter7.pkl
7,8,"(50,)",0.0010,100,0.833605,0.578799,0.066895,0.119929,0.733708,104.330171,0.312123,../checkpoints/mlp_gridsearch\mlp_iter8.pkl


Mejor combinación de hiperparámetros:
hidden_layer_sizes    (100, 50)
alpha                    0.0001
max_iter                     50
roc_auc                0.734623
f1_score               0.118109
recall                 0.065714
Name: 4, dtype: object
